# Transaction Foundation Model on Ray — Part 1: Setup

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

### Anyscale Technical Demo — Ray Data + Ray Train + Ray Serve

---

## What we're building

Transaction foundation models are the latest generation of transformer models — like LLMs, but instead of language, they are trained on financial transactions. This lets them recognize patterns, like fraud, that traditional ML techniques can't detect. Stripe, Visa, Mastercard, Nubank, and Revolut all run models like this in production, and NVIDIA published an open blueprint ([transaction-foundation-model](https://github.com/NVIDIA-AI-Blueprints/transaction-foundation-model)) so everyone else can build one too.

This series shows you how to build your own transaction foundation model, with performance and scalability that surpass NVIDIA's blueprint. The modeling is deliberately identical to theirs — we vendor their tokenizer, train their exact architecture with their recipe, and score fraud with their downstream setup. Ray is the only difference: it takes each stage from one machine to a cluster.

That difference is bigger than it sounds. NVIDIA's repo runs every stage on a single node, and it never actually trains the foundation model — the committed training config is a 30-step smoke run, and the real weights arrive as a file download. This template trains the same model from scratch, on all 19.5 million training transactions, in about two hours on eight GPUs — and every stage is code you can point at your own data.

## The result, up front

Fraud-detection results on the IBM TabFormer dataset, scored with NVIDIA's own evaluation protocol (average precision on their 100K stratified test set). These are from a real end-to-end run of this series, not projections:

| feature set | NVIDIA blueprint | this template |
|---|---|---|
| raw tabular features | 0.1238 | **0.1238** — reproduced exactly |
| FM embedding alone | 0.0123 | **0.0614** — ~5× better |
| fusion (raw + embedding) | 0.1755 | typical **0.136**, best draw **0.284** |

One honesty note on the fusion row, because it frames how Part 6 reports results: the test set contains only ~112 frauds, so any single fusion number is substantially luck of the draw. NVIDIA published one draw; we report the distribution. Their 0.1755 falls inside our range, and we clear it in about one draw in six — with a best draw well above it.

## Architecture

```
                     ┌────────────────────── Anyscale ───────────────────────┐
 IBM TabFormer ────► │ [Ray Data]   temporal split + NVIDIA-tokenizer corpus │
 (24M txns, raw CSV) │ [Ray Train]  next-token pretraining (Llama decoder)   │
                     │ [Ray Data]   batch embedding extraction (GPU workers) │
                     │ [XGBoost]    downstream fraud: raw vs FM vs fusion    │
                     │ [Ray Serve]  online embedding + fraud score           │
                     └───────────────────────────────────────────────────────┘
```

Every stage is the **same code** from a single machine to a multi-node cluster — you change `ScalingConfig`, not your program. One `SCALE` variable at the top of each notebook selects a config file under `configs/`, and diffing two of those files shows the complete difference between two scales.

## The series

We build the foundation model end-to-end, one stage per notebook:

| # | Notebook | Ray primitive |
|---|----------|---------------|
| **1** | **Setup** *(this notebook)* | — |
| 2 | Load & explore the data | Ray Data |
| 3 | Tokenize — build the pretrain corpus | Ray Data |
| 4 | Pretrain — next-token prediction | Ray Train |
| 5 | Batch embedding extraction | Ray Data |
| 6 | Downstream fraud — raw vs FM vs fusion | XGBoost |
| 7 | Online serving | Ray Serve |
| 8 | Run the whole pipeline as a job | Anyscale Jobs |
| 9 | Scaling up — the bottlenecks you'll hit | All of the above |

## What a full run needs

The series runs at two scales. `mini` runs everything on CPU with a small sample and a 2-layer model — it completes in minutes, proves the plumbing end-to-end, and is what CI runs. It does **not** produce the results above. For those you set `SCALE = "full"`, which needs:

- the IBM TabFormer dataset — a one-time ~2.3 GB download that Part 2 handles,
- GPU workers — 8×A10G for the ~2h pretraining and for embedding extraction; the downstream XGBoost also runs on a GPU, and Part 6 explains why the result genuinely depends on that,
- a CPU worker group in the cluster config, so the CPU-heavy data stages don't strand GPU nodes (Part 9 shows the failure mode when they do).

Nothing else changes between the two scales — same notebooks, same code, different config file.

---

## Step 1: Install dependencies

The cell below installs the template's dependencies and registers them on every cluster node, so the same imports resolve on workers as well as the head node. Two entries deserve a word: `xgboost` is pinned to **3.2.0** because the downstream fraud result is sensitive to an early-stopping behavior that changed in 3.3, and RAPIDS (`cudf`) is what NVIDIA's tokenizer uses on GPUs — at `mini` scale it falls back to scikit-learn on CPU automatically.

> In production you'd install from the generated `python_depset.lock`. Here we install from `requirements.txt` for portability.

In [1]:
!pip install -q -r requirements.txt

Successfully registered `ray, torch` and 12 other packages to be installed on all cluster nodes.
View and update dependencies here: https://console.anyscale.com/cld_g54aiirwj1s8t9ktgzikqur41k/prj_f1j47h9srml4cyg962id75ms2e/workspaces/expwrk_78mtwtucrd61tjxf851krarzwr?workspace-tab=dependencies


## Step 2: Attach to the cluster

In an Anyscale Workspace, Ray is already running — `ray.init()` **attaches** to the workspace's cluster rather than starting one. `working_dir` ships this template's `src/` package to every worker, so the notebook and remote tasks/actors all import the same code.

In [2]:
import sys, os
import logging


# Make the template's `src/` package importable from the notebook.
DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import ray

# In an Anyscale Workspace, Ray is already running — this attaches to it.
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR)

from src.utils import print_cluster_resources
print_cluster_resources()

Ray cluster resources:
  anyscale/cpu_only:true 1.0
  anyscale/node-group:head 1.0
  anyscale/provider:aws 1.0
  anyscale/region:us-west-2 1.0
  memory               34359738368.0
  object_store_memory  9600936345.0

Cluster nodes: 1
  10.0.190.216         alive=True  anyscale/node-group:head=1.0, object_store_memory=9600936345.0, anyscale/region:us-west-2=1.0, anyscale/provider:aws=1.0, memory=34359738368.0, anyscale/cpu_only:true=1.0


/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


You should see the head node and its resources. On a fresh or idle cluster the head node intentionally schedules no work — GPU and CPU worker nodes autoscale up when later stages request them, and scale back down when idle. You don't manage that; Ray does.

That's the whole setup: dependencies registered, cluster attached. Every notebook from here starts the same way and picks up where the previous one left off, using artifacts written to shared storage.

---

## Next

**Part 2 — Load & explore the data**: download the IBM TabFormer benchmark, rebuild NVIDIA's temporal train/val/test split from the raw CSV with Ray Data, and see in the data why a 0.1% fraud rate dictates how the whole series measures success.